# Day 095 Solution — Paper Trading Dashboard

In [ ]:
import pandas as pd, math
from dataclasses import dataclass, field

def _synthetic(n=252):
    prices = [100.0 * (1 + 0.3 * math.sin(i * 2 * math.pi / n)) for i in range(n)]
    dates  = pd.date_range("2023-01-01", periods=n, freq="B")
    close  = pd.Series(prices, index=dates)
    return pd.DataFrame({
        "Open":   close.shift(1).fillna(close.iloc[0]),
        "High":   close * 1.01,
        "Low":    close * 0.99,
        "Close":  close,
        "Volume": pd.Series([1_000_000 + i * 1_000 for i in range(n)], index=dates),
    })
@dataclass
class Trade:
    date:        object
    action:      str
    price:       float
    shares:      float
    cash_after:  float
    value_after: float
@dataclass
class PaperAccount:
    initial_cash: float = 10_000.0
    cash:         float = field(init=False)
    shares:       float = field(init=False)
    trades:       list  = field(init=False)

    def __post_init__(self):
        self.cash   = self.initial_cash
        self.shares = 0.0
        self.trades = []

    def portfolio_value(self, price):
        return self.cash + self.shares * float(price)

    def buy(self, date, price, fraction=1.0):
        price = float(price)
        if self.cash <= 0 or price <= 0:
            return None
        shares = (self.cash * fraction) / price
        cost   = shares * price
        if cost > self.cash:
            shares = self.cash / price
            cost   = shares * price
        self.cash   -= cost
        self.shares += shares
        t = Trade(date=date, action="BUY", price=price, shares=shares,
                  cash_after=self.cash,
                  value_after=self.portfolio_value(price))
        self.trades.append(t)
        return t

    def sell(self, date, price):
        price = float(price)
        if self.shares <= 0:
            return None
        proceeds    = self.shares * price
        sold_shares = self.shares
        self.cash  += proceeds
        self.shares = 0.0
        t = Trade(date=date, action="SELL", price=price, shares=sold_shares,
                  cash_after=self.cash,
                  value_after=self.portfolio_value(price))
        self.trades.append(t)
        return t
def run_paper_trader(df, signals, initial_cash=10_000.0, fraction=1.0):
    account = PaperAccount(initial_cash=initial_cash)
    eq_values   = []
    prev_signal = 0
    for i in range(len(df)):
        date  = df.index[i]
        price = float(df["Close"].iloc[i])
        sig   = int(signals.iloc[i])
        if sig == 1 and prev_signal == 0:
            account.buy(date, price, fraction=fraction)
        elif sig == 0 and prev_signal == 1:
            account.sell(date, price)
        eq_values.append(account.portfolio_value(price))
        prev_signal = sig
    if account.shares > 0:
        account.sell(df.index[-1], float(df["Close"].iloc[-1]))
    equity       = pd.Series(eq_values, index=df.index)
    total_return = float(equity.iloc[-1] / initial_cash - 1.0)
    peak         = equity.cummax()
    max_dd       = float(((equity - peak) / peak).min())
    return {
        "account":      account,
        "trades":       account.trades,
        "equity":       equity,
        "initial_cash": initial_cash,
        "final_value":  float(equity.iloc[-1]),
        "total_return": total_return,
        "max_drawdown": max_dd,
        "n_trades":     len(account.trades),
        "n_buys":       sum(1 for t in account.trades if t.action == "BUY"),
        "n_sells":      sum(1 for t in account.trades if t.action == "SELL"),
    }
def format_report(result):
    lines = [
        "=== Paper Trading Report ===",
        f"Initial cash :  ${result['initial_cash']:>12,.2f}",
        f"Final value  :  ${result['final_value']:>12,.2f}",
        f"Total return :  {result['total_return']:>12.2%}",
        f"Max drawdown :  {result['max_drawdown']:>12.2%}",
        f"Trades total :  {result['n_trades']:>12d}",
        f"  Buys       :  {result['n_buys']:>12d}",
        f"  Sells      :  {result['n_sells']:>12d}",
    ]
    if result["trades"]:
        first = result["trades"][0]
        last  = result["trades"][-1]
        lines.append(f"First trade  :  {first.action} @ ${first.price:,.2f}  ({first.date})")
        lines.append(f"Last trade   :  {last.action} @ ${last.price:,.2f}  ({last.date})")
    return "\n".join(lines)
def _sma_cross(df, fast=20, slow=50):
    c = df["Close"]
    return (c.rolling(fast).mean() > c.rolling(slow).mean()).fillna(False).astype(int)

def _apply_sl(signals, prices, stop_pct=0.05):
    result = signals.copy().astype(float)
    entry  = None
    for i in range(len(result)):
        if result.iloc[i] == 1:
            if entry is None:
                entry = float(prices.iloc[i])
            elif float(prices.iloc[i]) <= entry * (1 - stop_pct):
                result.iloc[i] = 0
                entry = None
        else:
            entry = None
    return result.astype(int)

def _apply_dd(signals, prices, limit=-0.20):
    peak = prices.cummax()
    dd   = (prices - peak) / peak
    r    = signals.copy().astype(int)
    r[dd < limit] = 0
    return r


In [ ]:
df = _synthetic(n=252)

strategies = [
    ("Always-Flat", pd.Series(0, index=df.index)),
    ("Always-Long", pd.Series(1, index=df.index)),
    ("SMA-cross",   _sma_cross(df)),
]

results = {}
for label, sig in strategies:
    r = run_paper_trader(df, sig, initial_cash=10_000.0)
    results[label] = r

# Assertions
flat  = results["Always-Flat"]
long_ = results["Always-Long"]
sma   = results["SMA-cross"]

assert flat["n_trades"]  == 0,  f"always-flat should have 0 trades, got {flat['n_trades']}"
assert (flat["equity"] == 10_000.0).all(), "always-flat equity should be constant"
assert long_["n_buys"]  == 1,   f"always-long: 1 buy"
assert long_["n_sells"] == 1,   f"always-long: 1 sell (forced)"
for label, r in results.items():
    assert r["max_drawdown"] <= 1e-9, f"{label}: max_drawdown should be ≤ 0"
    assert abs(r["final_value"] - r["equity"].iloc[-1]) < 1e-9
    diff = abs(r["n_buys"] - r["n_sells"])
    assert diff <= 1, f"{label}: |buys-sells| should be 0 or 1"

for label, r in results.items():
    print(f"\n{'='*40}")
    print(format_report(r))

print("\nSolution smoke-test passed.")
